In [0]:

# =========================================================
# PHASE 3: SILVER LAYER
# STEP 1: DATA QUALITY PROFILING
# =========================================================

from pyspark.sql import functions as F
from src.transformations.silver import transform_silver

BRONZE_TABLE = (
    "workspace.video_game_sales.bronze_video_game_sales"
)

# Read Bronze table
bronze_df = spark.table(BRONZE_TABLE)

total_records = bronze_df.count()

print("TOTAL BRONZE RECORDS:", total_records)

# 1. NULL VALUE ANALYSIS
null_expressions = [
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in bronze_df.columns
]

null_report = bronze_df.agg(*null_expressions)

print("\n--- NULL VALUE REPORT ---")
display(null_report)

# 2. DUPLICATE ANALYSIS
print("\n--- DUPLICATE ANALYSIS ---")

full_duplicates = (
    total_records - bronze_df.dropDuplicates().count()
)

duplicate_ranks = (
    total_records - bronze_df.select("Rank").distinct().count()
)

print("Full duplicate rows:", full_duplicates)
print("Duplicate Rank values:", duplicate_ranks)

# 3. YEAR ANALYSIS
print("\n--- YEAR DISTRIBUTION ---")

bronze_df.groupBy("Year") \
    .count() \
    .orderBy("Year") \
    .show(50, truncate=False)

# 4. SALES VALIDATION
sales_columns = [
    "NA_Sales",
    "EU_Sales",
    "JP_Sales",
    "Other_Sales",
    "Global_Sales"
]

print("\n--- SALES VALIDATION ---")

sales_report = bronze_df.agg(
    *[
        F.sum(
            F.when(F.col(c) < 0, 1).otherwise(0)
        ).alias(f"{c}_negative")
        for c in sales_columns
    ]
)

display(sales_report)

# 5. MISSING DIMENSION VALUES
dimension_columns = [
    "Name",
    "Platform",
    "Year",
    "Genre",
    "Publisher"
]

print("\n--- MISSING DIMENSIONS ---")

dimension_report = bronze_df.agg(
    *[
        F.sum(
            F.when(
                F.col(c).isNull() |
                (F.trim(F.col(c)) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in dimension_columns
    ]
)

display(dimension_report)

# 6. BUSINESS KEY CHECK
print("\n--- RANK UNIQUENESS ---")

bronze_df.groupBy("Rank") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(20, truncate=False)

print("\nDATA QUALITY PROFILING COMPLETED")


In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.video_game_sales.bronze_video_game_sales"
)

for column in bronze_df.columns:
    null_count = bronze_df.filter(
        F.col(column).isNull()
    ).count()

    print(f"{column}: {null_count}")

In [0]:
total = bronze_df.count()

full_duplicates = total - bronze_df.dropDuplicates().count()

duplicate_ranks = (
    total - bronze_df.select("Rank").distinct().count()
)

print("Total records:", total)
print("Full duplicates:", full_duplicates)
print("Duplicate ranks:", duplicate_ranks)

In [0]:
bronze_df.groupBy("Year") \
    .count() \
    .orderBy("Year") \
    .show(50, truncate=False)

In [0]:
sales_columns = [
    "NA_Sales", "EU_Sales",
    "JP_Sales", "Other_Sales",
    "Global_Sales"
]

for column in sales_columns:
    print(
        column,
        "Negative values:",
        bronze_df.filter(F.col(column) < 0).count()
    )

In [0]:
dimensions = [
    "Name", "Platform", "Year",
    "Genre", "Publisher"
]

for column in dimensions:
    missing = bronze_df.filter(
        F.col(column).isNull() |
        (F.trim(F.col(column)) == "")
    ).count()

    print(f"{column}: {missing}")

In [0]:

# =========================================================
# SILVER: ADDITIONAL DATA QUALITY INVESTIGATION
# =========================================================

from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.video_game_sales.bronze_video_game_sales"
)

# 1. YEAR VALIDATION

year_profile = bronze_df.withColumn(
    "year_numeric",
    F.when(
        F.col("Year").rlike(r"^\d{4}$"),
        F.col("Year").cast("int")
    )
).withColumn(
    "year_status",
    F.when(
        F.col("year_numeric").isNotNull(),
        "VALID"
    ).otherwise("MISSING_OR_INVALID")
)

print("--- YEAR QUALITY ---")

year_profile.groupBy("year_status") \
    .count() \
    .show()

print("Invalid year values:")

year_profile.filter(
    F.col("year_status") == "MISSING_OR_INVALID"
).groupBy("Year").count().show()

# 2. WHITESPACE VALIDATION

text_columns = [
    "Name", "Platform", "Genre", "Publisher"
]

print("--- WHITESPACE CHECK ---")

for column in text_columns:
    count = bronze_df.filter(
        F.col(column) != F.trim(F.col(column))
    ).count()

    print(f"{column}: {count}")

# 3. REGIONAL VS GLOBAL SALES RECONCILIATION

sales_df = bronze_df.withColumn(
    "regional_sales_total",
    F.round(
        F.col("NA_Sales") +
        F.col("EU_Sales") +
        F.col("JP_Sales") +
        F.col("Other_Sales"),
        2
    )
).withColumn(
    "sales_difference",
    F.round(
        F.abs(
            F.col("Global_Sales") -
            F.col("regional_sales_total")
        ),
        2
    )
)

print("--- SALES RECONCILIATION ---")

sales_df.select(
    F.count("*").alias("total_records"),
    F.sum(
        F.when(
            F.col("sales_difference") > 0,
            1
        ).otherwise(0)
    ).alias("records_with_difference"),
    F.max("sales_difference").alias("maximum_difference")
).show()

print("Largest sales differences:")

sales_df.orderBy(
    F.desc("sales_difference")
).select(
    "Rank",
    "Name",
    "Global_Sales",
    "regional_sales_total",
    "sales_difference"
).show(10, truncate=False)

print("ADDITIONAL PROFILING COMPLETED")


In [0]:
# =========================================================
# PHASE 3: SILVER LAYER
# DATA CLEANING AND STANDARDIZATION
# =========================================================

from pyspark.sql import functions as F
from src.transformations.silver import transform_silver


# =========================================================
# TABLE CONFIGURATION
# =========================================================

BRONZE_TABLE = (
    "workspace.video_game_sales.bronze_video_game_sales"
)

SILVER_TABLE = (
    "workspace.video_game_sales.silver_video_game_sales"
)


# =========================================================
# 1. READ BRONZE TABLE
# =========================================================

bronze_df = spark.table(BRONZE_TABLE)

print("Bronze table loaded:", BRONZE_TABLE)
print("Bronze record count:", bronze_df.count())


# =========================================================
# 2. APPLY SILVER TRANSFORMATION
# =========================================================

silver_df = transform_silver(bronze_df)

print("Silver transformation completed.")
print("Transformed records:", silver_df.count())


# =========================================================
# 3. INSPECT TRANSFORMED DATA
# =========================================================

print("Silver DataFrame schema:")
silver_df.printSchema()

display(silver_df.limit(10))


# =========================================================
# 4. WRITE SILVER DELTA TABLE
# =========================================================

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print("Silver table created:", SILVER_TABLE)


# =========================================================
# 5. VALIDATE RECORD PRESERVATION
# =========================================================

silver_count = spark.table(SILVER_TABLE).count()
bronze_count = bronze_df.count()

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)

assert bronze_count == silver_count, (
    "Record count mismatch between Bronze and Silver layers."
)

print("SILVER RECORD RECONCILIATION PASSED")


# =========================================================
# 6. VALIDATE YEAR QUALITY
# =========================================================

display(
    spark.table(SILVER_TABLE)
    .groupBy("year_status")
    .count()
    .orderBy("year_status")
)


# =========================================================
# 7. VALIDATE SALES RECONCILIATION
# =========================================================

display(
    spark.table(SILVER_TABLE)
    .groupBy("sales_reconciliation_status")
    .count()
    .orderBy("sales_reconciliation_status")
)


# =========================================================
# 8. VALIDATE SALES DIFFERENCES
# =========================================================

display(
    spark.table(SILVER_TABLE)
    .filter(F.col("sales_difference") > 0)
    .select(
        "rank",
        "name",
        "na_sales",
        "eu_sales",
        "jp_sales",
        "other_sales",
        "global_sales",
        "regional_sales_total",
        "sales_difference",
        "sales_reconciliation_status"
    )
    .orderBy(F.desc("sales_difference"))
    .limit(20)
)


# =========================================================
# 9. SILVER LAYER SUMMARY
# =========================================================

silver_summary = spark.table(SILVER_TABLE)

print("=================================================")
print("SILVER LAYER SUMMARY")
print("=================================================")

print("Bronze records:", bronze_count)
print("Silver records:", silver_count)

print(
    "Valid years:",
    silver_summary.filter(
        F.col("year_status") == "VALID"
    ).count()
)

print(
    "Invalid or missing years:",
    silver_summary.filter(
        F.col("year_status") == "MISSING_OR_INVALID"
    ).count()
)

print(
    "Sales matches:",
    silver_summary.filter(
        F.col("sales_reconciliation_status") == "MATCH"
    ).count()
)

print(
    "Sales differences:",
    silver_summary.filter(
        F.col("sales_reconciliation_status") == "DIFFERENCE_DETECTED"
    ).count()
)

print("=================================================")
print("SILVER LAYER EXECUTION COMPLETED")
print("=================================================")

In [0]:
silver_df = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

print("TOTAL SILVER RECORDS:", silver_df.count())

print("\n--- YEAR QUALITY ---")

silver_df.groupBy("year_status") \
    .count() \
    .show()

print("\n--- SALES RECONCILIATION ---")

silver_df.groupBy("sales_reconciliation_status") \
    .count() \
    .show()

print("\n--- NULL YEAR VALUES ---")

print(
    "Missing years:",
    silver_df.filter(F.col("year").isNull()).count()
)

print("\n--- SILVER SCHEMA ---")

silver_df.printSchema()

print("\n--- SAMPLE RECORDS ---")

display(silver_df.limit(10))

In [0]:
# =========================================================
# SAFE SILVER TRANSFORMATION SMOKE TEST
# READ-ONLY: DOES NOT MODIFY ANY DELTA TABLE
# =========================================================

from pyspark.sql import functions as F
from src.transformations.silver import transform_silver

# 1. Define table names explicitly
BRONZE_TABLE = "workspace.video_game_sales.bronze_video_game_sales"
SILVER_TABLE = "workspace.video_game_sales.silver_video_game_sales"

# 2. Read existing tables
bronze_test_df = spark.table(BRONZE_TABLE)
existing_silver_df = spark.table(SILVER_TABLE)

# 3. Apply the updated transformation
transformed_silver_df = transform_silver(bronze_test_df)

# 4. Compare record counts
bronze_count = bronze_test_df.count()
existing_silver_count = existing_silver_df.count()
transformed_count = transformed_silver_df.count()

print("Bronze records:", bronze_count)
print("Existing Silver records:", existing_silver_count)
print("New transformed records:", transformed_count)

# 5. Validate record preservation
assert bronze_count == transformed_count, (
    "FAIL: Transformation changed the Bronze record count."
)

assert existing_silver_count == transformed_count, (
    "FAIL: New transformation count differs from existing Silver."
)

# 6. Inspect schema
print("\nTransformed Silver schema:")
transformed_silver_df.printSchema()

# 7. Inspect sample records
display(
    transformed_silver_df.select(
        "rank",
        "name",
        "year_raw",
        "year",
        "year_status",
        "regional_sales_total",
        "sales_difference",
        "sales_reconciliation_status"
    ).limit(10)
)

print("\nSAFE SMOKE TEST PASSED")

In [0]:
# =========================================================
# COMPARE EXISTING SILVER WITH NEW TRANSFORMATION
# READ-ONLY VALIDATION
# =========================================================

from pyspark.sql import functions as F

comparison_columns = [
    "rank",
    "name",
    "platform",
    "year_raw",
    "genre",
    "publisher",
    "na_sales",
    "eu_sales",
    "jp_sales",
    "other_sales",
    "global_sales",
    "year",
    "year_status",
    "regional_sales_total",
    "sales_difference",
    "sales_reconciliation_status"
]

old_df = existing_silver_df.select(*comparison_columns)
new_df = transformed_silver_df.select(*comparison_columns)

# Compare both directions
old_only = old_df.exceptAll(new_df).count()
new_only = new_df.exceptAll(old_df).count()

print("Records present only in existing Silver:", old_only)
print("Records present only in new transformation:", new_only)

assert old_only == 0 and new_only == 0, (
    "VALIDATION FAILED: Existing and new Silver records differ."
)

print("SILVER TRANSFORMATION EQUIVALENCE PASSED")